# CO₂ mondial : historique, scénarios et cartes

Données utilisées :
- `../data/preproc/world_total_co2_emissions.xlsx` (historical/current/stated/netzero).
- `../data/electricity/World CO2 Emissions_COPY.xlsx` (par pays, pour carte et top émetteurs).
- `../data/preproc/total_co2_emissions.xlsx` (régions/scénarios pour vérifier cohérence).

Objectifs :
- Visualiser la trajectoire mondiale selon scénarios, avec pointillés après 2025.
- Quantifier écarts 2035/2050 (Δ MtCO₂, % vs current), CAGR de décroissance.
- Carte des émissions par pays (dernière année), top 10 émetteurs et part cumulée.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9,5)

root_el = Path('..') / 'data' / 'electricity'
root_pre = Path('..') / 'data' / 'preproc'


## Carte CO₂ (dernière année) + top 10


In [2]:
co2_xls = pd.ExcelFile(root_el / 'World CO2 Emissions_COPY.xlsx')
df_co2 = pd.read_excel(co2_xls, sheet_name=co2_xls.sheet_names[0])
df_co2 = df_co2.rename(columns={df_co2.columns[0]: 'Country'})
years = [c for c in df_co2.columns if str(c).strip().isdigit()]
df_co2 = df_co2[['Country'] + years]
last_year = max(int(y) for y in years)
df_latest = df_co2[['Country', str(last_year)]].copy()
df_latest.columns = ['Country','CO2_Mt']

fig = px.choropleth(df_latest, locations='Country', locationmode='country names', color='CO2_Mt',
                    color_continuous_scale='Reds', title=f"Émissions CO₂ (Mt) - {last_year}")
fig.show()

top10 = df_latest.sort_values('CO2_Mt', ascending=False).head(10)
top10['Part_%'] = top10['CO2_Mt']/top10['CO2_Mt'].sum()*100
top10


ValueError: max() iterable argument is empty

## Trajectoires globales par scénario (pointillés après 2025)


In [3]:
global_co2 = pd.read_excel(root_pre / 'world_total_co2_emissions.xlsx')
long = global_co2.melt(var_name='horizon', value_name='MtCO2', value_vars=[c for c in global_co2.columns if c != global_co2.columns[0]])
long[['year','scenario']] = long['horizon'].str.extract(r'(\d{4})_(.*)')
long['year'] = long['year'].fillna(long['horizon'].where(long['horizon'].str.isdigit(), None))
long = long.dropna(subset=['year'])
long['year'] = long['year'].astype(int)
long['scenario'] = long['scenario'].fillna('historical').str.lower()
long['line_dash'] = np.where(long['year']>2025, 'dash', 'solid')
fig = px.line(long, x='year', y='MtCO2', color='scenario', line_dash='line_dash', markers=True,
              title='CO₂ global : historical vs current/stated/netzero (pointillés >2025)')
fig.show()


## Écarts 2035/2050 et CAGR de décroissance


In [4]:
pivot = long.pivot_table(index=['year','scenario'], values='MtCO2', aggfunc='sum').reset_index()
base_year = 2024
current = pivot[pivot['scenario']=='current'].set_index('year')['MtCO2']
subset = pivot[pivot['year'].isin([2035,2050])].copy()
subset['delta_vs_current'] = subset.apply(lambda r: r['MtCO2'] - current.get(r['year'], np.nan), axis=1)
subset['pct_vs_current'] = subset.apply(lambda r: r['MtCO2']/current.get(r['year'], np.nan)*100 if current.get(r['year'], np.nan) else np.nan, axis=1)
subset


,year,scenario,MtCO2,delta_vs_current,pct_vs_current
6,2035,current,38503,0,100.000000
7,2035,nze,17606,-20897,45.726307
8,2035,stated,35244,-3259,91.535724
12,2050,current,37779,0,100.000000
13,2050,nze,0,-37779,0.000000
14,2050,stated,29629,-8150,78.427169


In [5]:
# CAGR 2024->2035 et 2024->2050
cagr_rows = []
for sc in long['scenario'].unique():
    sc_data = long[long['scenario']==sc]
    if base_year in sc_data['year'].values:
        base = sc_data.loc[sc_data['year']==base_year,'MtCO2'].values[0]
        for target in [2035,2050]:
            if target in sc_data['year'].values and base>0:
                val = sc_data.loc[sc_data['year']==target,'MtCO2'].values[0]
                yrs = target-base_year
                cagr_rows.append({'scenario':sc,'target_year':target,'CAGR':(val/base)**(1/yrs)-1})
pd.DataFrame(cagr_rows)


,scenario,target_year,CAGR
0,current,2035,0.000831
1,current,2050,-0.000379
2,stated,2035,-0.007184
3,stated,2050,-0.009678
4,nze,2035,-0.067891
5,nze,2050,-1.000000


## Vérification régions/scénarios (optionnel)


In [6]:
reg = pd.read_excel(root_pre / 'total_co2_emissions.xlsx')
reg_long = reg.melt(id_vars='Region', var_name='horizon', value_name='MtCO2')
reg_long[['year','scenario']] = reg_long['horizon'].str.extract(r'(\d{4})_(.*)')
reg_long = reg_long.dropna(subset=['year'])
reg_long['year'] = reg_long['year'].astype(int)
reg_long['scenario'] = reg_long['scenario'].fillna('historical').str.lower()
reg_long['line_dash'] = np.where(reg_long['year']>2025, 'dash', 'solid')
fig = px.line(reg_long, x='year', y='MtCO2', color='Region', line_dash='line_dash', facet_col='scenario', facet_col_wrap=2,
              title='Émissions CO₂ par région et scénario')
fig.update_layout(height=500)
fig.show()


## Lecture rapide
- Carte + top 10 : localise les “poches” d’émissions actuelles.
- Courbes : netzero vs current/stated met en évidence l’effort de réduction. Pointillés >2025 = projections.
- Écarts 2035/2050 + CAGR : quantifient la décroissance requise.
